<a href="https://colab.research.google.com/github/7vika-lgtm/LLMQ-A/blob/main/3_llm_ask_my_notes_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ask My Notes: Retrieval-Augmented Q&A with a Large Language Model
**Skill:** Large Language Models (LLMs), prompt engineering, retrieval-augmented generation (RAG).

**What you build:** a question-answering system that answers *only* from your documents, cites its sources, and says "I don't know" when the answer is not there.

**How it works**
1. Split documents into small chunks
2. Find the chunks most similar to the question (TF-IDF search)
3. Give those chunks to the LLM as context, with strict instructions
4. Check the system on test questions, including one it should refuse

**Setup:** you need an Anthropic API key. In Colab click the key icon (Secrets) in the left bar, add a secret named `ANTHROPIC_API_KEY`, and switch on notebook access. Usage for this notebook costs very little.

## 1. Setup

In [1]:
!pip install -q anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 24.4 MB/s eta 0:00:00


In [ ]:
import os, getpass, re
import pandas as pd
from anthropic import Anthropic
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

try:
    from google.colab import userdata
    API_KEY = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    API_KEY = os.environ.get("ANTHROPIC_API_KEY") or getpass.getpass("Anthropic API key: ")

client = Anthropic(api_key=API_KEY)
MODEL = "claude-sonnet-5-5"   # change if you want a different Claude model

## 2. Your documents
This sample is a made-up cafe staff handbook. To use your own notes, run the optional upload cell below it.

In [ ]:
DOCS = {
 "opening-hours.txt": "Northwind Cafe opens at 7:00 AM on weekdays and 8:30 AM on weekends. The cafe closes at 6:00 PM Monday to Saturday and 4:00 PM on Sunday. Staff must arrive 30 minutes before opening to prepare the espresso machines and bakery display. On public holidays the cafe opens at 9:00 AM and closes at 3:00 PM.",
 "refunds.txt": "Customers can return unopened packaged goods within 14 days with a receipt. Drinks and fresh food cannot be refunded, but a mistake in the order must be remade for free. Refunds above 50 dollars require approval from the shift manager. Refunds are paid back to the original payment method.",
 "allergies.txt": "Always ask customers about allergies before taking a food order. The kitchen uses separate boards and knives for nut-free orders. Oat milk is the only dairy-free milk that is also nut-free. If a customer reports a severe reaction, call emergency services immediately and then notify the shift manager. All allergen information is listed in the blue folder behind the counter.",
 "leave.txt": "Full-time staff receive 20 days of paid leave per year. Leave requests must be submitted at least 14 days in advance through the staff portal. Sick leave does not count toward annual leave but requires a doctor's note after 3 consecutive days.",
}
print({k: len(v.split()) for k, v in DOCS.items()}, "words per document")

In [ ]:
# OPTIONAL: upload your own .txt files (replaces the sample documents)
# from google.colab import files
# up = files.upload()
# DOCS = {name: data.decode("utf-8", errors="ignore") for name, data in up.items()}

## 3. Chunking
Long documents are cut into overlapping pieces so the search can find the exact passage that matters.

In [ ]:
def chunk_text(text, size=40, overlap=10):
    words = text.split()
    step = size - overlap
    return [" ".join(words[i:i + size]) for i in range(0, max(1, len(words) - overlap), step)]

chunks = []   # list of dicts: source + text
for name, text in DOCS.items():
    for piece in chunk_text(text):
        chunks.append({"source": name, "text": piece})
print(len(chunks), "chunks")
pd.DataFrame(chunks).head()

## 4. Retrieval
TF-IDF turns each chunk into a vector of weighted pieces. We use character n-grams so that word variants such as "refund" and "refunds" still match. We rank chunks by cosine similarity to the question.

In [ ]:
vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5))   # character n-grams match word variants (refund/refunds, approve/approval)
matrix = vectorizer.fit_transform([c["text"] for c in chunks])

def retrieve(question, k=3):
    scores = cosine_similarity(vectorizer.transform([question]), matrix)[0]
    top = scores.argsort()[::-1][:k]
    return [{**chunks[i], "score": float(scores[i])} for i in top if scores[i] > 0]

for r in retrieve("When does the cafe open on weekends?"):
    print(round(r["score"], 2), r["source"], "|", r["text"][:80], "...")

## 5. Prompt + generation
The system prompt controls the behaviour: stay inside the context, cite sources, and refuse politely when the answer is missing.

In [ ]:
STRICT = (
    "You answer questions using ONLY the numbered context passages provided.\n"
    "Rules:\n"
    "- Cite the passages you used like [1] or [2].\n"
    "- If the context does not contain the answer, reply exactly: I don't know based on the documents.\n"
    "- Be concise (1 to 3 sentences). Never use outside knowledge."
)

def ask(question, system=STRICT, k=3, show_sources=False):
    hits = retrieve(question, k)
    context = "\n\n".join(f"[{i+1}] ({h['source']}) {h['text']}" for i, h in enumerate(hits)) or "(no relevant passages found)"
    resp = client.messages.create(
        model=MODEL, max_tokens=400, system=system,
        messages=[{"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}],
    )
    answer = resp.content[0].text.strip()
    if show_sources:
        print("SOURCES:", [h["source"] for h in hits])
    return answer, hits

answer, _ = ask("Who has to approve a refund over 50 dollars?", show_sources=True)
print(answer)

## 6. Try your own questions

In [ ]:
for q in ["What time does the cafe close on Sunday?",
          "What should I do if a customer has a severe allergic reaction?",
          "What is the cafe's wifi password?"]:
    a, _ = ask(q)
    print("Q:", q); print("A:", a); print()

## 7. Evaluate the system
A tiny test set: each question has a keyword that must appear in the retrieved text (retrieval check) and in the answer (answer check). The last question has no answer in the documents, so the correct behaviour is to refuse.

In [ ]:
tests = [
 ("What time does the cafe open on Sunday?", "8:30"),
 ("Who must approve an 80 dollar refund?", "shift manager"),
 ("Which milk is safe for a nut allergy?", "oat"),
 ("How many days notice is needed for leave?", "14"),
 ("What is the owner's favourite colour?", None),     # unanswerable on purpose
]
rows = []
for q, key in tests:
    a, hits = ask(q)
    retrieved = " ".join(h["text"] for h in hits).lower()
    if key is None:
        rows.append({"question": q, "retrieval": "n/a", "answer ok": "don't know" in a.lower(), "answer": a})
    else:
        rows.append({"question": q, "retrieval": key.lower() in retrieved, "answer ok": key.lower() in a.lower(), "answer": a})
report = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 90)
report

## 8. Prompt engineering experiment
Same question and context, two different system prompts. Compare how strict instructions change the output on a question the documents cannot answer.

In [ ]:
LOOSE = "You are a helpful assistant. Use the context if it helps."
q = "How many sick days do part-time staff get?"
for label, sysmsg in [("LOOSE", LOOSE), ("STRICT", STRICT)]:
    a, _ = ask(q, system=sysmsg)
    print(f"--- {label} ---"); print(a); print()

**What to notice:** the loose prompt may fill the gap with plausible-sounding guesses (a *hallucination*). The strict prompt keeps the model grounded.

## 9. Conclusions & limitations
- Write what worked and what failed in your tests.
- **Limitations:** TF-IDF matches spelling, not meaning (it misses true synonyms like "quit" and "resign"); the evaluation has only 5 questions; chunk size affects quality.
- **Next steps:** use embeddings instead of TF-IDF, add a re-ranking step, load PDFs, add chat history, deploy as a Streamlit or Gradio app, and score answers automatically with an LLM judge.

## References
- Lewis et al. (2020), *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*.
- Anthropic documentation: https://docs.claude.com